In [0]:
from pyspark.sql import functions as F

SILVER       = "capstone_project_dev.silver.validated_metadata"
GOLD_METRICS = "capstone_project_dev.gold.governance_metrics"
GOLD_NONCOMPLIANT = "capstone_project_dev.gold.non_compliant_tables"
GOLD_TABLE_COUNTS = "capstone_project_dev.gold.table_column_counts"

print("Setup done")


In [0]:
silver = spark.table(SILVER)

print(f"Rows loaded : {silver.count():,}")
print(f"Cols loaded : {len(silver.columns)}")

In [0]:
total_tables  = silver.select("table_name").distinct().count()
total_columns = silver.select("table_name", "column_name").distinct().count()

# Structural compliance — tables with column_count >= threshold (mean - 1 stddev)
structural_compliance_pct = (
    silver.select("table_name", "rule06_below_col_standard")
          .distinct()
          .filter(F.col("rule06_below_col_standard") == False)
          .count()
    / total_tables * 100
)

# PII coverage — PII rows correctly marked Confidential
pii_total = silver.filter(F.col("pii_flag") == True).count()
pii_covered = silver.filter(
    (F.col("pii_flag") == True) &
    (F.col("security_classification") == "Confidential")
).count()
pii_coverage_pct = (pii_covered / pii_total * 100) if pii_total > 0 else 0

# Certification coverage — distinct columns with non-null certification
cert_covered = silver.select("table_name", "column_name", "certification_level") \
    .distinct() \
    .filter(F.col("certification_level").isNotNull()) \
    .count()
cert_coverage_pct = (cert_covered / total_columns * 100)

# CDEs missing a steward
cde_missing_steward = silver.filter(
    (F.col("critical_data_element_flag") == True) &
    (F.col("data_steward").isNull())
).count()

# Metadata completeness — across 4 key fields, deduplicated
fields_to_check = ["column_desc", "term_name", "data_steward", "security_classification"]
total_possible  = total_columns * len(fields_to_check)
total_filled    = sum(
    silver.select("table_name", "column_name", f)
          .distinct()
          .filter(F.col(f).isNotNull())
          .count()
    for f in fields_to_check
)
metadata_completeness_pct = (total_filled / total_possible * 100)

# Column compliance only — excludes RULE-06 (table-level rule)
column_compliant = silver.select("table_name", "column_name", "column_compliance") \
    .distinct() \
    .filter(F.col("column_compliance") == "COMPLIANT") \
    .count()
column_compliance_pct = (column_compliant / total_columns * 100)

# Table compliance — structural standard only (RULE-06)
table_compliant = silver.select("table_name", "table_compliance") \
    .distinct() \
    .filter(F.col("table_compliance") == "COMPLIANT") \
    .count()
table_compliance_pct = (table_compliant / total_tables * 100)

# Overall — fails if either table or column compliance fails
overall_compliant = silver.select("table_name", "column_name", "compliance_flag") \
    .distinct() \
    .filter(F.col("compliance_flag") == "COMPLIANT") \
    .count()
overall_compliance_pct = (overall_compliant / total_columns * 100)

print(f"Total tables              : {total_tables}")
print(f"Total columns             : {total_columns:,}")
print(f"Structural compliance %   : {structural_compliance_pct:.1f}%")
print(f"Table compliance %        : {table_compliance_pct:.1f}%")
print(f"Column compliance %       : {column_compliance_pct:.1f}%")
print(f"PII coverage %            : {pii_coverage_pct:.1f}%")
print(f"Certification coverage %  : {cert_coverage_pct:.1f}%")
print(f"CDE missing steward       : {cde_missing_steward:,}")
print(f"Metadata completeness %   : {metadata_completeness_pct:.1f}%")
print(f"Overall compliance %      : {overall_compliance_pct:.1f}%")

In [0]:
from pyspark.sql.functions import greatest

# Get the dynamic threshold from Silver
threshold_val = silver.select("col_standard_threshold").first()[0]

non_compliant_summary = (
    silver.filter(F.col("compliance_flag") == "NON-COMPLIANT")
    .groupBy("table_name")
    .agg(
        F.count("*").alias("non_compliant_row_count"),
        F.sum(F.col("rule01_mandatory_null").cast("int")).alias("rule01_violations"),
        F.sum(F.col("rule02_pii_flag_invalid").cast("int")).alias("rule02_violations"),
        F.sum(F.col("rule03_security_invalid").cast("int")).alias("rule03_violations"),
        F.sum(F.col("rule04_cert_level_invalid").cast("int")).alias("rule04_violations"),
        F.sum(F.col("rule05_pii_not_confidential").cast("int")).alias("rule05_violations"),
        F.sum(F.col("rule06_below_col_standard").cast("int")).alias("rule06_violations"),
        F.sum(F.col("rule07_cde_no_steward").cast("int")).alias("rule07_violations"),
        F.sum(F.col("rule08_invalid_count_exceeds_total").cast("int")).alias("rule08_violations"),
    )
    # Find worst rule by picking the one with highest violation count
    # Only considers rules with count > 0 — avoids zero-max mislabeling
    # Tiebreak priority: rule03 → rule07 → rule04 → rule06 → rule05 → rule02 → rule08 → rule01
    .withColumn("worst_rule",
        F.when((F.col("rule03_violations") > 0) & (F.col("rule03_violations") == greatest(
            F.col("rule01_violations"), F.col("rule02_violations"), F.col("rule03_violations"),
            F.col("rule04_violations"), F.col("rule05_violations"), F.col("rule06_violations"),
            F.col("rule07_violations"), F.col("rule08_violations")
        )), "rule03")
        .when((F.col("rule07_violations") > 0) & (F.col("rule07_violations") == greatest(
            F.col("rule01_violations"), F.col("rule02_violations"), F.col("rule03_violations"),
            F.col("rule04_violations"), F.col("rule05_violations"), F.col("rule06_violations"),
            F.col("rule07_violations"), F.col("rule08_violations")
        )), "rule07")
        .when((F.col("rule04_violations") > 0) & (F.col("rule04_violations") == greatest(
            F.col("rule01_violations"), F.col("rule02_violations"), F.col("rule03_violations"),
            F.col("rule04_violations"), F.col("rule05_violations"), F.col("rule06_violations"),
            F.col("rule07_violations"), F.col("rule08_violations")
        )), "rule04")
        .when((F.col("rule06_violations") > 0) & (F.col("rule06_violations") == greatest(
            F.col("rule01_violations"), F.col("rule02_violations"), F.col("rule03_violations"),
            F.col("rule04_violations"), F.col("rule05_violations"), F.col("rule06_violations"),
            F.col("rule07_violations"), F.col("rule08_violations")
        )), "rule06")
        .when((F.col("rule05_violations") > 0) & (F.col("rule05_violations") == greatest(
            F.col("rule01_violations"), F.col("rule02_violations"), F.col("rule03_violations"),
            F.col("rule04_violations"), F.col("rule05_violations"), F.col("rule06_violations"),
            F.col("rule07_violations"), F.col("rule08_violations")
        )), "rule05")
        .when((F.col("rule02_violations") > 0) & (F.col("rule02_violations") == greatest(
            F.col("rule01_violations"), F.col("rule02_violations"), F.col("rule03_violations"),
            F.col("rule04_violations"), F.col("rule05_violations"), F.col("rule06_violations"),
            F.col("rule07_violations"), F.col("rule08_violations")
        )), "rule02")
        .when((F.col("rule08_violations") > 0) & (F.col("rule08_violations") == greatest(
            F.col("rule01_violations"), F.col("rule02_violations"), F.col("rule03_violations"),
            F.col("rule04_violations"), F.col("rule05_violations"), F.col("rule06_violations"),
            F.col("rule07_violations"), F.col("rule08_violations")
        )), "rule08")
        .when(F.col("rule01_violations") > 0, "rule01")
        .otherwise(None)
    )
    # Dynamic suggested_fix — injects actual column count and threshold
    .withColumn("suggested_fix",
        F.when(F.col("worst_rule") == "rule03",
            F.lit("Fix security_classification — allowed values: Internal, Confidential, Public"))
        .when(F.col("worst_rule") == "rule07",
            F.lit("Assign a data steward to all critical data elements per GOV-111 RULE-07"))
        .when(F.col("worst_rule") == "rule04",
            F.lit("Add or fix certification_level — allowed values: Registered, Certified, Documented"))
        .when(F.col("worst_rule") == "rule06",
            F.concat(
                F.lit("Table has "),
                F.col("rule06_violations").cast("string"),
                F.lit(f" columns vs the {threshold_val:.1f}-column standard — review schema or document as approved exception")
            ))
        .when(F.col("worst_rule") == "rule05",
            F.lit("Reclassify PII columns as Confidential per GOV-111 RULE-05"))
        .when(F.col("worst_rule") == "rule02",
            F.lit("Fix pii_flag — value is null and must be True or False"))
        .when(F.col("worst_rule") == "rule08",
            F.lit("Fix invalid_record_count — value exceeds total_record_count"))
        .when(F.col("worst_rule") == "rule01",
            F.lit("Populate mandatory fields: column_id, table_id, column_name, table_name"))
        .otherwise(F.lit("No violations found"))
    )
    .orderBy("non_compliant_row_count", ascending=False)
)

non_compliant_summary.display()

In [0]:
# Table column counts — answers "how many columns does table X have?"
# Direct backing for Objective 1 chatbot and dashboard bar chart
threshold_val = silver.select("col_standard_threshold").first()[0]

table_column_counts = (
    silver.groupBy("table_name")
    .agg(
        F.countDistinct("column_name").alias("column_count")
    )
    .withColumn("threshold", F.lit(round(threshold_val, 1)))
    .withColumn("is_compliant",
        F.when(F.col("column_count") >= F.col("threshold"), True)
        .otherwise(False)
    )
    .withColumn("compliance_status",
        F.when(F.col("is_compliant"), "COMPLIANT")
        .otherwise("NON-COMPLIANT")
    )
    .orderBy("column_count", ascending=True)
)

table_column_counts.display()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
from datetime import datetime

# Build single-row summary metrics table
gold_data = [(
    int(total_tables),
    int(total_columns),
    round(float(structural_compliance_pct), 2),
    round(float(table_compliance_pct), 2),
    round(float(column_compliance_pct), 2),
    round(float(pii_coverage_pct), 2),
    round(float(cert_coverage_pct), 2),
    int(cde_missing_steward),
    round(float(metadata_completeness_pct), 2),
    round(float(overall_compliance_pct), 2),
    datetime.now().strftime("%Y-%m-%d %H:%M:%S")
)]

gold_schema = StructType([
    StructField("total_tables",                LongType(),   False),
    StructField("total_columns",               LongType(),   False),
    StructField("structural_compliance_pct",   DoubleType(), False),
    StructField("table_compliance_pct",        DoubleType(), False),
    StructField("column_compliance_pct",       DoubleType(), False),
    StructField("pii_coverage_pct",            DoubleType(), False),
    StructField("certification_coverage_pct",  DoubleType(), False),
    StructField("cde_missing_steward_count",   LongType(),   False),
    StructField("metadata_completeness_pct",   DoubleType(), False),
    StructField("overall_compliance_pct",      DoubleType(), False),
    StructField("calculated_at",               StringType(), False),
    
])

gold_df = spark.createDataFrame(gold_data, schema=gold_schema)

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_METRICS)

print(f"Gold metrics table written: {GOLD_METRICS}")

# Write non-compliant summary
non_compliant_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_NONCOMPLIANT)

# Write table column counts
table_column_counts.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TABLE_COUNTS)

print(f"Table column counts written: {GOLD_TABLE_COUNTS}")
print(f"Non-compliant summary written: {GOLD_NONCOMPLIANT}")

In [0]:
print("=== GOLD METRICS ===")
spark.table(GOLD_METRICS).display()

print("=== NON-COMPLIANT TABLES ===")
spark.table(GOLD_NONCOMPLIANT).display()